# Notebook 13 — Process Soil-to-Leaf Nutrient Data

This notebook processes the raw soil and leaf nutrient datasets (`All.csv` and `COCONUTTREE_ExportFeatures_TableToExcel.csv`). It standardizes columns, concatenates both datasets to increase rows, drops nulls, engineers nutrient ratios, and exports the cleaned dataset for ML training.

In [1]:
import os
import sys
import subprocess

def install_if_missing(module_name, package_name=None):
    if package_name is None:
        package_name = module_name
    try:
        __import__(module_name)
    except ImportError:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

install_if_missing('pandas')
install_if_missing('numpy')

import pandas as pd
import numpy as np
from pathlib import Path

# Ensure project root is in the path
current_dir = Path(os.getcwd())
if current_dir.name == 'notebooks':
    sys.path.append(str(current_dir.parent))
else:
    sys.path.append(str(current_dir))
    
import config

RAW_DIR = config.BASE_DIR / 'raw'
PROCESSED_DIR = config.PROCESSED_DIR

## 1. Load Raw Datasets
Loading `All.csv` and `COCONUTTREE_ExportFeatures_TableToExcel.csv`.

In [2]:
all_csv_path = RAW_DIR / 'All.csv'
large_csv_path = RAW_DIR / 'COCONUTTREE_ExportFeatures_TableToExcel.csv'

df1 = pd.DataFrame()
df2 = pd.DataFrame()

if all_csv_path.exists():
    df1 = pd.read_csv(all_csv_path)
    print(f"Loaded {len(df1)} rows from All.csv.")

if large_csv_path.exists():
    df2 = pd.read_csv(large_csv_path)
    print(f"Loaded {len(df2)} rows from COCONUTTREE Export Features.")


Loaded 135 rows from All.csv.
Loaded 5702 rows from COCONUTTREE Export Features.


## 2. Standardize Columns & Concatenate
We standardize feature names to `soil_n`, `soil_p`, `soil_k` and targets to `leaf_n`, `leaf_p`, `leaf_k` across both datasets and merge them.

In [3]:
# Rename map for All.csv
rename_map_1 = {
    'Tree No': 'tree_id',
    'Soil N': 'soil_n',
    'Soil P': 'soil_p',
    'Soil K': 'soil_k',
    'Leaf N': 'leaf_n',
    'Leaf P': 'leaf_p',
    'Leaf K': 'leaf_k'
}

# Rename map for the large CSV
rename_map_2 = {
    'tree Id': 'tree_id',
    'Nitrogen_soil': 'soil_n',
    'Phosphors_soil': 'soil_p',
    'Potassium_soil': 'soil_k',
    'Nitrogen_leaf': 'leaf_n',
    'Phosphors_leaf': 'leaf_p',
    'Potassium_leaf': 'leaf_k'
}

if not df1.empty:
    df1 = df1.rename(columns=rename_map_1)
if not df2.empty:
    df2 = df2.rename(columns=rename_map_2)

critical_cols = ['soil_n', 'soil_p', 'soil_k', 'leaf_n', 'leaf_p', 'leaf_k']

# Ensure only relevant columns are kept before concat
if not df1.empty:
    df1 = df1[['tree_id'] + critical_cols if 'tree_id' in df1.columns else critical_cols]
if not df2.empty:
    df2 = df2[['tree_id'] + critical_cols if 'tree_id' in df2.columns else critical_cols]

# Concatenate both datasets vertically to increase the row count!
df = pd.concat([df1, df2], ignore_index=True)
print(f"Total rows after concatenation: {len(df)}")

# Drop any rows with NaN in critical columns
df = df.dropna(subset=critical_cols)
print(f"Shape after dropping nulls: {df.shape}")

Total rows after concatenation: 5837
Shape after dropping nulls: (5815, 7)


## 3. Feature Engineering

In [4]:
epsilon = 1e-6
df['np_ratio'] = df['soil_n'] / (df['soil_p'] + epsilon)
df['nk_ratio'] = df['soil_n'] / (df['soil_k'] + epsilon)
df['pk_ratio'] = df['soil_p'] / (df['soil_k'] + epsilon)
df['npk_sum'] = df['soil_n'] + df['soil_p'] + df['soil_k']

print(df[['soil_n', 'soil_p', 'soil_k', 'np_ratio', 'nk_ratio', 'pk_ratio', 'npk_sum']].head())
print("\nDataset summary statistics:")
print(df.describe())

   soil_n   soil_p    soil_k  np_ratio  nk_ratio  pk_ratio   npk_sum
0  0.0159  0.34302  0.062942  0.046353  0.252610  5.449693  0.421862
1  0.0335  0.15298  0.065793  0.218981  0.509165  2.325136  0.252273
2  0.0182  0.15068  0.086126  0.120785  0.211316  1.749509  0.255006
4  0.0218  0.11085  0.111790  0.196660  0.195007  0.991583  0.244440
5  0.0124  0.11184  0.140943  0.110872  0.087978  0.793507  0.265183

Dataset summary statistics:
           tree_id       soil_n       soil_p       soil_k       leaf_n  \
count  5815.000000  5815.000000  5815.000000  5815.000000  5815.000000   
mean   2781.496991     0.020427     0.224632     0.088338     1.518205   
std    1648.248872     0.007946     0.097264     0.033393     0.306621   
min       0.000000     0.000600     0.053050     0.037693     0.577200   
25%    1351.500000     0.014700     0.158349     0.069543     1.312693   
50%    2756.000000     0.018132     0.206749     0.080060     1.530913   
75%    4209.500000     0.025369     0.2

## 4. Export Processed Data

In [5]:
out_path = PROCESSED_DIR / 'soil_leaf_processed.csv'
df.to_csv(out_path, index=False)
print(f"Successfully saved processed data ({len(df)} rows) to {out_path}")

Successfully saved processed data (5815 rows) to C:\Users\yoshi\Desktop\PP2\V1\backend\Coconut_Yield_Prediction\processed\soil_leaf_processed.csv
